In [2]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

ds = load_dataset("Tobi-Bueck/customer-support-tickets",
                  data_files="dataset-tickets-multi-lang-4-20k.csv")
df = ds["train"].to_pandas()

df = df[df["language"] == "en"].copy()
df = df.dropna(subset=["body", "queue"])
texts = df["body"].tolist()
labels = df["queue"].tolist()

print(f"Loaded {len(texts)} documents.")

c:\Users\sande\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 11922 documents.


In [3]:
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(len(X_train), len(X_val), len(X_test))

8345 1788 1789


In [4]:
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

tfidf_vectorizer = TfidfVectorizer()
train_features = tfidf_vectorizer.fit_transform(X_train)
val_features = tfidf_vectorizer.transform(X_val)
test_features = tfidf_vectorizer.transform(X_test)

label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(y_train)
val_labels = label_encoder.transform(y_val)
test_labels = label_encoder.transform(y_test)

train_x = torch.tensor(train_features.toarray(), dtype=torch.float32)
val_x = torch.tensor(val_features.toarray(), dtype=torch.float32)
test_x = torch.tensor(test_features.toarray(), dtype=torch.float32)

train_y = torch.tensor(train_labels, dtype=torch.long)
val_y = torch.tensor(val_labels, dtype=torch.long)
test_y = torch.tensor(test_labels, dtype=torch.long)

print(train_x.shape, val_x.shape, test_x.shape)

torch.Size([8345, 4751]) torch.Size([1788, 4751]) torch.Size([1789, 4751])


In [5]:
import torch.nn as nn

class FeedforwardClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.hidden_layer = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.output_layer = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.hidden_layer(x)
        x = self.activation(x)
        x = self.output_layer(x)
        return x

model = FeedforwardClassifier(
    input_size=train_x.shape[1],
    hidden_size=256,
    num_classes=len(label_encoder.classes_),
)
print(model)

FeedforwardClassifier(
  (hidden_layer): Linear(in_features=4751, out_features=256, bias=True)
  (activation): ReLU()
  (output_layer): Linear(in_features=256, out_features=10, bias=True)
)


In [6]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score

torch.manual_seed(42)

train_batches = DataLoader(TensorDataset(train_x, train_y), batch_size=64, shuffle=True)

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(20):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_batches:
        optimizer.zero_grad()
        scores = model(batch_x)
        loss = loss_function(scores, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    with torch.no_grad():
        train_predictions = model(train_x).argmax(dim=1)

    train_accuracy = accuracy_score(train_y, train_predictions)

    print(f"Epoch {epoch + 1}: loss = {total_loss / len(train_batches):.4f}, "
          f"train acc = {train_accuracy:.3f}")

Epoch 1: loss = 1.9207, train acc = 0.411
Epoch 2: loss = 1.6003, train acc = 0.488
Epoch 3: loss = 1.3822, train acc = 0.602
Epoch 4: loss = 1.1795, train acc = 0.683
Epoch 5: loss = 1.0015, train acc = 0.732
Epoch 6: loss = 0.8579, train acc = 0.789
Epoch 7: loss = 0.7380, train acc = 0.831
Epoch 8: loss = 0.6427, train acc = 0.851
Epoch 9: loss = 0.5623, train acc = 0.884
Epoch 10: loss = 0.4931, train acc = 0.895
Epoch 11: loss = 0.4356, train acc = 0.916
Epoch 12: loss = 0.3855, train acc = 0.934
Epoch 13: loss = 0.3399, train acc = 0.947
Epoch 14: loss = 0.3018, train acc = 0.950
Epoch 15: loss = 0.2690, train acc = 0.965
Epoch 16: loss = 0.2383, train acc = 0.971
Epoch 17: loss = 0.2136, train acc = 0.978
Epoch 18: loss = 0.1889, train acc = 0.982
Epoch 19: loss = 0.1672, train acc = 0.986
Epoch 20: loss = 0.1486, train acc = 0.990


In [9]:

from sklearn.metrics import classification_report, f1_score, accuracy_score

model.eval()
with torch.no_grad():
    val_predictions = model(val_x).argmax(dim=1)

print(f"Val accuracy: {accuracy_score(val_y, val_predictions):.3f}")
print(f"Val macro F1: {f1_score(val_y, val_predictions, average='macro'):.3f}")
print(classification_report(val_y, val_predictions, target_names=label_encoder.classes_))

Val accuracy: 0.458
Val macro F1: 0.419
                                 precision    recall  f1-score   support

           Billing and Payments       0.76      0.71      0.73       195
               Customer Service       0.34      0.38      0.36       279
                General Inquiry       0.60      0.24      0.34        25
                Human Resources       0.53      0.26      0.35        31
                     IT Support       0.37      0.36      0.36       208
                Product Support       0.40      0.46      0.43       334
          Returns and Exchanges       0.32      0.26      0.29        87
            Sales and Pre-Sales       0.41      0.18      0.25        50
Service Outages and Maintenance       0.71      0.48      0.57        67
              Technical Support       0.50      0.52      0.51       512

                       accuracy                           0.46      1788
                      macro avg       0.49      0.39      0.42      1788
         

In [13]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_y.numpy()),
    y=train_y.numpy(),
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

for name, weight in zip(label_encoder.classes_, class_weights_tensor):
    print(f"{name}: {weight:.3f}")

Billing and Payments: 0.916
Customer Service: 0.641
General Inquiry: 7.072
Human Resources: 5.795
IT Support: 0.857
Product Support: 0.534
Returns and Exchanges: 2.050
Sales and Pre-Sales: 3.613
Service Outages and Maintenance: 2.701
Technical Support: 0.349


In [14]:
torch.manual_seed(42)

model = FeedforwardClassifier(
    input_size=train_x.shape[1],
    hidden_size=256,
    num_classes=len(label_encoder.classes_),
)

train_batches = DataLoader(TensorDataset(train_x, train_y), batch_size=64, shuffle=True)

loss_function = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(20):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_batches:
        optimizer.zero_grad()
        scores = model(batch_x)
        loss = loss_function(scores, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch + 1}: loss = {total_loss / len(train_batches):.4f}")

Epoch 1: loss = 2.1978
Epoch 2: loss = 1.8066
Epoch 3: loss = 1.4027
Epoch 4: loss = 1.0802
Epoch 5: loss = 0.8491
Epoch 6: loss = 0.6862
Epoch 7: loss = 0.5725
Epoch 8: loss = 0.4869
Epoch 9: loss = 0.4225
Epoch 10: loss = 0.3686
Epoch 11: loss = 0.3267
Epoch 12: loss = 0.2916
Epoch 13: loss = 0.2605
Epoch 14: loss = 0.2372
Epoch 15: loss = 0.2119
Epoch 16: loss = 0.1900
Epoch 17: loss = 0.1715
Epoch 18: loss = 0.1541
Epoch 19: loss = 0.1401
Epoch 20: loss = 0.1276


In [15]:
from sklearn.metrics import classification_report

model.eval()
with torch.no_grad():
    val_predictions = model(val_x).argmax(dim=1)

print(f"Val accuracy: {accuracy_score(val_y, val_predictions):.3f}")
print(f"Val macro F1: {f1_score(val_y, val_predictions, average='macro'):.3f}")
print(classification_report(val_y, val_predictions, target_names=label_encoder.classes_))

Val accuracy: 0.455
Val macro F1: 0.419
                                 precision    recall  f1-score   support

           Billing and Payments       0.74      0.73      0.73       195
               Customer Service       0.37      0.42      0.39       279
                General Inquiry       0.32      0.24      0.27        25
                Human Resources       0.35      0.26      0.30        31
                     IT Support       0.37      0.35      0.36       208
                Product Support       0.40      0.41      0.41       334
          Returns and Exchanges       0.32      0.38      0.35        87
            Sales and Pre-Sales       0.37      0.30      0.33        50
Service Outages and Maintenance       0.57      0.55      0.56        67
              Technical Support       0.50      0.48      0.49       512

                       accuracy                           0.45      1788
                      macro avg       0.43      0.41      0.42      1788
         

In [16]:
print(df["priority"].value_counts())

priority
medium    4951
high      4571
low       2400
Name: count, dtype: int64


In [17]:
priority_labels = df["priority"].tolist()

p_train, p_temp = train_test_split(
    priority_labels, test_size=0.3, random_state=42, stratify=labels
)
p_val, p_test = train_test_split(
    p_temp, test_size=0.5, random_state=42, stratify=y_temp
)

priority_encoder = LabelEncoder()
train_priority = torch.tensor(priority_encoder.fit_transform(p_train), dtype=torch.long)
val_priority = torch.tensor(priority_encoder.transform(p_val), dtype=torch.long)
test_priority = torch.tensor(priority_encoder.transform(p_test), dtype=torch.long)

print(len(p_train), len(p_val), len(p_test))

8345 1788 1789


In [18]:
torch.manual_seed(42)

priority_model = FeedforwardClassifier(
    input_size=train_x.shape[1],
    hidden_size=256,
    num_classes=len(priority_encoder.classes_),
)

priority_batches = DataLoader(TensorDataset(train_x, train_priority), batch_size=64, shuffle=True)

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(priority_model.parameters(), lr=0.001)

for epoch in range(20):
    priority_model.train()
    total_loss = 0

    for batch_x, batch_y in priority_batches:
        optimizer.zero_grad()
        scores = priority_model(batch_x)
        loss = loss_function(scores, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch + 1}: loss = {total_loss / len(priority_batches):.4f}")

Epoch 1: loss = 1.0413
Epoch 2: loss = 0.9290
Epoch 3: loss = 0.7880
Epoch 4: loss = 0.6676
Epoch 5: loss = 0.5634
Epoch 6: loss = 0.4712
Epoch 7: loss = 0.3910
Epoch 8: loss = 0.3197
Epoch 9: loss = 0.2548
Epoch 10: loss = 0.2039
Epoch 11: loss = 0.1636
Epoch 12: loss = 0.1290
Epoch 13: loss = 0.1017
Epoch 14: loss = 0.0818
Epoch 15: loss = 0.0659
Epoch 16: loss = 0.0537
Epoch 17: loss = 0.0444
Epoch 18: loss = 0.0368
Epoch 19: loss = 0.0308
Epoch 20: loss = 0.0263


In [19]:
priority_model.eval()
with torch.no_grad():
    val_priority_predictions = priority_model(val_x).argmax(dim=1)

print(f"Val accuracy: {accuracy_score(val_priority, val_priority_predictions):.3f}")
print(f"Val macro F1: {f1_score(val_priority, val_priority_predictions, average='macro'):.3f}")
print(classification_report(val_priority, val_priority_predictions, target_names=priority_encoder.classes_))

Val accuracy: 0.573
Val macro F1: 0.559
              precision    recall  f1-score   support

        high       0.58      0.62      0.60       665
         low       0.49      0.47      0.48       389
      medium       0.61      0.58      0.59       734

    accuracy                           0.57      1788
   macro avg       0.56      0.56      0.56      1788
weighted avg       0.57      0.57      0.57      1788



In [20]:
from captum.attr import IntegratedGradients

model.eval()
integrated_gradients = IntegratedGradients(model)

feature_names = tfidf_vectorizer.get_feature_names_out()
print(len(feature_names))

4751


In [21]:
sample_index = 0
sample_input = test_x[sample_index].unsqueeze(0)
sample_input.requires_grad_()

predicted_class = model(sample_input).argmax(dim=1).item()

attributions, _ = integrated_gradients.attribute(
    sample_input, target=predicted_class, return_convergence_delta=True
)

attributions = attributions.squeeze(0).detach().numpy()

top_indices = attributions.argsort()[-10:][::-1]
print(f"Predicted queue: {label_encoder.classes_[predicted_class]}")
for i in top_indices:
    print(f"{feature_names[i]}: {attributions[i]:.4f}")

Predicted queue: Returns and Exchanges
multiple: 1.7886
unsuccessful: 1.0812
where: 1.0042
failed: 0.8277
reinstall: 0.7606
an: 0.4895
attempts: 0.4249
promptly: 0.3080
assist: 0.2640
can: 0.1855


In [22]:
import numpy as np

queue_scores = {name: np.zeros(len(feature_names)) for name in label_encoder.classes_}
queue_counts = {name: 0 for name in label_encoder.classes_}

model.eval()
for i in range(len(test_x)):
    ticket = test_x[i].unsqueeze(0)
    ticket.requires_grad_()
    predicted = model(ticket).argmax(dim=1).item()

    attributions = integrated_gradients.attribute(ticket, target=predicted)
    queue_name = label_encoder.classes_[predicted]
    queue_scores[queue_name] += attributions.squeeze(0).detach().numpy()
    queue_counts[queue_name] += 1

for queue_name in label_encoder.classes_:
    if queue_counts[queue_name] == 0:
        continue
    average_scores = queue_scores[queue_name] / queue_counts[queue_name]
    top_words = [feature_names[j] for j in average_scores.argsort()[-10:][::-1]]
    print(f"{queue_name} ({queue_counts[queue_name]} tickets): {', '.join(top_words)}")

Billing and Payments (200 tickets): billing, payment, subscriptions, discounts, charges, invoices, invoice, pricing, my, support
Customer Service (301 tickets): data, for, integration, details, on, any, decisions, please, strategies, integrating
General Inquiry (19 tickets): receive, data, driving, minimal, it, effective, situation, and, providers, incorporating
Human Resources (37 tickets): employee, gaming, employees, systems, training, detected, broader, within, request, potential
IT Support (197 tickets): network, tools, drivers, need, marketing, of, restarting, website, or, believe
Product Support (309 tickets): api, project, settings, timelines, might, resolve, rates, integration, data, problem
Returns and Exchanges (117 tickets): assistance, products, has, drop, management, require, your, module, user, multiple
Sales and Pre-Sales (42 tickets): solutions, sales, investment, required, financial, that, immediate, utilized, machine, studies
Service Outages and Maintenance (43 ticke